# Actualización Selectiva del Pipeline bajo Drift — Physionet (Sepsis, Logistic Regression)

Evalúa estrategias de respuesta ante drift sintético sobre un pipeline
de **dos etapas** entrenado en datos de referencia:

| ID      | Estrategia               | Descripción                                                                            |
| ------- | ------------------------ | -------------------------------------------------------------------------------------- |
| **A1**  | **Defer**                | No hacer nada — usar pipeline original                                                 |
| **A2**  | **Actualizar features**  | Reajustar Stage 1 (QT + imputación + StandardScaler) con datos nuevos, mantener modelo |
| **A2c** | **Features correctivos** | Corregir datos drifteados hacia referencia, aplicar Stage 1 original y mantener modelo |
| **A3**  | **Actualizar modelo**    | Mantener Stage 1 original, reentrenar Logistic Regression con datos nuevos             |
| **A4**  | **Reentrenar todo**      | Reajustar Stage 1 y reentrenar Logistic Regression — referencia de costo máximo        |

**Métrica de recuperación:**
$$\text{Rec}\% = \frac{\text{AUC}_{\text{estrategia}}}{\text{AUC}_{\text{baseline}}} \times 100$$

**Costo:** tiempo relativo a A4 (1.00×) por condición.

**Acción óptima:** estrategia con menor costo total residual + cómputo.


## Pipeline de dos etapas

$$\hat{y} = f_2 \left( f_1(X) \right)$$

Donde:

- $f_1$ — **Stage 1**: `QuantileTransformer` ajustado sobre VITALS, imputación por mediana y `StandardScaler` ajustados sobre todas las variables numéricas del conjunto de referencia.
- $f_2$ — **Stage 2**: `LogisticRegression` ajustado sobre la salida de $f_1$.

Cuando ocurre drift, el diagnóstico determina si actualizar $f_1$ (covariate), $f_2$ (concept) o ambas (both).


## 1. Imports y configuración


In [ ]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    average_precision_score,
    precision_recall_curve,
    classification_report,
)

from darl.actions import run_a1, run_a2, run_a2_corrective, run_a3, run_a4
from darl.data.get_dataset import load_dataset, find_project_root
from darl.drift import DriftInjector
from darl.pipeline import apply_stage1, fit_stage1, make_logreg
from darl.visualization.drift_plots import plot_numeric_drift_grid

SEED = 42
np.random.seed(SEED)

# Variables de signos vitales — Stage 1 (QuantileTransformer)
VITALS = ["HR", "SBP", "MAP", "Resp", "Temp"]

# Configuración de drift por variable
NUM_CFG = {
    "HR": "high",
    "SBP": "low",
    "MAP": "low",
    "Resp": "high",
    "Temp": "extreme",
}

LABEL_COL = "SepsisLabel"

# Usamos el set de entrenamiento completo para reentrenar (A3/A4)

print("Imports OK")

Imports OK


## 2. Carga del dataset Physionet


In [ ]:
dset = load_dataset("physionet")

X_train, y_train, _, _ = dset.get_pandas(split="train")
X_test, y_test, _, _ = dset.get_pandas(split="id_test")

df_train = X_train.copy()
df_train[LABEL_COL] = y_train.values

df_target = X_test.copy()
df_target[LABEL_COL] = y_test.values

NUMERIC_COLS = (
    df_train.drop(columns=[LABEL_COL])
    .select_dtypes(include=["number"])
    .columns.tolist()
)

print(f"Train  : {df_train.shape}")
print(f"Target : {df_target.shape}")
print(f"Features numericas: {len(NUMERIC_COLS)}")
print(f"Prevalencia Sepsis Train : {y_train.mean():.4f}")
print(f"Prevalencia Sepsis Target: {y_test.mean():.4f}")
print(f"VITALS presentes: {[c for c in VITALS if c in df_target.columns]}")

## 3. Entrenamiento del Pipeline Baseline (dos etapas)

- **Stage 1**: `QuantileTransformer` ajustado sobre VITALS de `df_train` + imputación por mediana + `StandardScaler` ajustado sobre `NUMERIC_COLS`.
- **Stage 2**: `LogisticRegression` ajustada sobre datos transformados por Stage 1.


In [ ]:
# ─── Stage 1: QuantileTransformer sobre VITALS + imputacion + StandardScaler ───
qt_ref, imputer_ref, scaler_ref = fit_stage1(
    df_train,
    VITALS,
    NUMERIC_COLS,
    seed=SEED,
)

# ─── Stage 2: Logistic Regression sobre datos transformados ───
df_train_t = apply_stage1(
    df_train,
    qt_ref,
    imputer_ref,
    scaler_ref,
    VITALS,
    NUMERIC_COLS,
)
X_tr = df_train_t[NUMERIC_COLS].values
y_tr = df_train_t[LABEL_COL].values

neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
print(f"Clase 0: {neg:,}  |  Clase 1: {pos:,}")

model_ref = make_logreg(seed=SEED)
model_ref.fit(X_tr, y_tr)
print("Pipeline baseline Logistic Regression entrenado.")

Clase 0: 1,109,051  |  Clase 1: 13,248
Pipeline baseline Logistic Regression entrenado.


## 4. Evaluación Baseline

Umbral óptimo calculado maximizando F1 sobre el test set limpio.


In [ ]:
df_target_t = apply_stage1(
    df_target,
    qt_ref,
    imputer_ref,
    scaler_ref,
    VITALS,
    NUMERIC_COLS,
)
X_te = df_target_t[NUMERIC_COLS].values
y_te = df_target_t[LABEL_COL].values

y_prob_base = model_ref.predict_proba(X_te)[:, 1]

AUC_BASE = roc_auc_score(y_te, y_prob_base)
AUPR_BASE = average_precision_score(y_te, y_prob_base)

prec_arr, rec_arr, thresh_arr = precision_recall_curve(y_te, y_prob_base)
f1_arr = 2 * prec_arr[:-1] * rec_arr[:-1] / (prec_arr[:-1] + rec_arr[:-1] + 1e-9)
BEST_THRESHOLD = float(thresh_arr[f1_arr.argmax()])

y_pred_base = (y_prob_base >= BEST_THRESHOLD).astype(int)
F1_BASE = f1_score(y_te, y_pred_base)

print(f"AUC-ROC baseline         : {AUC_BASE:.4f}")
print(f"AUC-PR  baseline         : {AUPR_BASE:.4f}")
print(f"Umbral optimo (F1)       : {BEST_THRESHOLD:.4f}")
print(f"F1-score (umbral optimo) : {F1_BASE:.4f}")
print()
print(classification_report(y_te, y_pred_base))

AUC-ROC baseline         : 0.6428
AUC-PR  baseline         : 0.0258
Umbral optimo (F1)       : 0.6977
F1-score (umbral optimo) : 0.0678

              precision    recall  f1-score   support

           0       0.99      0.97      0.98    138598
           1       0.05      0.11      0.07      1690

    accuracy                           0.96    140288
   macro avg       0.52      0.54      0.52    140288
weighted avg       0.98      0.96      0.97    140288



## 5. Ajuste del DriftInjector


In [7]:
inj = DriftInjector(random_state=SEED)
inj.fit(df_train, numeric_cols=VITALS)
print(f"DriftInjector ajustado sobre VITALS: {inj._numeric_cols}")

DriftInjector ajustado sobre VITALS: ['HR', 'SBP', 'MAP', 'Resp', 'Temp']


## 6. Definición de Estrategias A1–A4

Cada estrategia recibe los datos drifteados y el conjunto de entrenamiento
drifteado (para reajuste), y devuelve métricas + tiempo de ejecución.

|     | Stage 1 (QT)   | Stage 2 (LogReg) |
| --- | -------------- | ---------------- |
| A1  | original       | original         |
| A2  | **reajustado** | original         |
| A3  | original       | **reentrenado**  |
| A4  | **reajustado** | **reentrenado**  |


In [ ]:
print("Estrategias A1-A4 + A2c importadas desde darl.actions.")

Estrategias A1-A4 + A2c definidas para Logistic Regression.


## 7. Experimento Principal

Para cada combinación (tipo de drift × severidad):

1. Se aplica drift sobre `df_target` (datos de evaluación).
2. Se aplica el mismo drift sobre **todo** `df_train` (datos de reentrenamiento para A3/A4).
3. Se evalúan las 4 estrategias midiendo AUC y tiempo.

> **Comparación justa:** A1 usa el modelo original entrenado con 1.1M filas.
> A3/A4 reentrenan sobre **1.1M filas con drift**, no un subconjunto.
> Así la diferencia de AUC refleja el tipo de drift, no el tamaño del dataset.


In [ ]:
EXPERIMENT_GRID = [
    # (drift_type,   severidad_label, severity_value)
    ("covariate", "Leve", 0.2),
    ("covariate", "Moderada", 0.5),
    ("covariate", "Severa", 0.8),
    ("concept", "Leve", 0.2),
    ("concept", "Moderada", 0.5),
    ("concept", "Severa", 0.8),
    ("both", "Moderada", 0.5),
    ("both", "Severa", 0.8),
]

# Datos de reentrenamiento = todo df_train (misma escala que el modelo original A1)
# Comparacion justa: A1 usa 1.1M filas originales,
# A3/A4 reentrenan con 1.1M filas drifteadas.
df_train_retrain = df_train  # sin subsampling
print(f"Set de reentrenamiento: {len(df_train_retrain):,} filas")

DRIFT_LABEL = {
    "covariate": "Covariate shift",
    "concept": "Concept drift",
    "both": "Covariate + Concept",
}

rows = []

for drift_type, sev_label, sev in EXPERIMENT_GRID:
    print(f"\n[{DRIFT_LABEL[drift_type]} — {sev_label} (sev={sev})]")

    # ── Drift sobre df_target y df_train ──
    if drift_type == "covariate":
        df_d_target, _ = inj.transform(
            df_target,
            drift_severity=sev,
            drift_type="covariate",
            numeric_drift_config=NUM_CFG,
        )
        df_d_train, _ = inj.transform(
            df_train_retrain,
            drift_severity=sev,
            drift_type="covariate",
            numeric_drift_config=NUM_CFG,
        )
    elif drift_type == "concept":
        # Concept drift via feature relationship inversion (cambio en P(Y|X))
        df_d_target = df_target.copy()
        df_d_train = df_train_retrain.copy()
        for col in VITALS:
            med_target = df_target[col].median()
            df_d_target[col] = df_target[col] + sev * (
                2 * med_target - 2 * df_target[col]
            )
            med_train = df_train_retrain[col].median()
            df_d_train[col] = df_train_retrain[col] + sev * (
                2 * med_train - 2 * df_train_retrain[col]
            )
    else:  # both
        # 1. Aplicamos covariate drift (Beta-mixture)
        df_d_target, _ = inj.transform(
            df_target,
            drift_severity=sev,
            drift_type="covariate",
            numeric_drift_config=NUM_CFG,
        )
        df_d_train, _ = inj.transform(
            df_train_retrain,
            drift_severity=sev,
            drift_type="covariate",
            numeric_drift_config=NUM_CFG,
        )
        # 2. Aplicamos concept drift via feature inversion (cambio en P(Y|X))
        for col in VITALS:
            med_target = df_d_target[col].median()
            df_d_target[col] = df_d_target[col] + sev * (
                2 * med_target - 2 * df_d_target[col]
            )
            med_train = df_d_train[col].median()
            df_d_train[col] = df_d_train[col] + sev * (
                2 * med_train - 2 * df_d_train[col]
            )

    # ── A1: No hacer nada ──
    m1, t1 = run_a1(
        df_d_target,
        qt_ref,
        imputer_ref,
        scaler_ref,
        model_ref,
        VITALS,
        NUMERIC_COLS,
        LABEL_COL,
        BEST_THRESHOLD,
    )
    print(f'  A1 Defer       : AUC={m1["auc"]:.4f}  t={t1:.2f}s')

    # ── A2: Reajustar Stage 1 (QT + imputer + scaler), mantener modelo ──
    m2, t2 = run_a2(
        df_d_target,
        df_d_train,
        model_ref,
        VITALS,
        NUMERIC_COLS,
        LABEL_COL,
        BEST_THRESHOLD,
        seed=SEED,
    )
    print(f'  A2 Feat update : AUC={m2["auc"]:.4f}  t={t2:.2f}s')

    # ── A2c: Corregir drift hacia referencia, usar Stage 1 original, mantener modelo ──
    m2c, t2c = run_a2_corrective(
        df_d_target,
        df_d_train,
        df_train,
        qt_ref,
        imputer_ref,
        scaler_ref,
        model_ref,
        VITALS,
        NUMERIC_COLS,
        LABEL_COL,
        BEST_THRESHOLD,
    )
    print(f'  A2 Corrective  : AUC={m2c["auc"]:.4f}  t={t2c:.2f}s')

    # ── A3: Mantener Stage 1, reentrenar Logistic Regression con datos drifteados ──
    m3, t3 = run_a3(
        df_d_target,
        df_d_train,
        qt_ref,
        imputer_ref,
        scaler_ref,
        VITALS,
        NUMERIC_COLS,
        LABEL_COL,
        BEST_THRESHOLD,
        seed=SEED,
    )
    print(f'  A3 Model update: AUC={m3["auc"]:.4f}  t={t3:.2f}s')

    # ── A4: Reajustar Stage 1 + reentrenar Logistic Regression ──
    m4, t4 = run_a4(
        df_d_target,
        df_d_train,
        VITALS,
        NUMERIC_COLS,
        LABEL_COL,
        BEST_THRESHOLD,
        seed=SEED,
    )
    print(f'  A4 Retrain all : AUC={m4["auc"]:.4f}  t={t4:.2f}s')

    # ── Calcular metricas de la tabla ──
    rec = lambda m: round(m["auc"] / AUC_BASE * 100, 1)

    # Costos relativos medidos por condicion. A4 define costo 1.00x.
    c1 = 0.00
    c4 = 1.00
    c2 = t2 / t4 if t4 > 0 else 0.00
    c2c = t2c / t4 if t4 > 0 else 0.00
    c3 = t3 / t4 if t4 > 0 else 0.00

    r1, r2, r2c, r3, r4 = rec(m1), rec(m2), rec(m2c), rec(m3), rec(m4)

    # ── Accion optima via CARA-inspired Total Cost Minimization ──
    # Basado en: Mahadevan & Mathioudakis (2024), 'Cost-aware retraining for ML'
    # TotalCost(Ax) = Ψ(Ax) + λ · κ(Ax)
    # Donde:
    #   Ψ(Ax) = (100 - Rec%(Ax))  -> degradacion residual de AUC tras aplicar Ax
    #   κ(Ax)  = costo relativo de computo medido contra A4 por condicion
    #   λ = 10 -> un reentrenamiento completo equivale a 10 puntos de Rec%
    # La accion optima minimiza el costo total (sin restriccion por tipo de drift).
    # Este es un benchmark neutral: la hipotesis de DARL se evalua a posteriori.
    LAMBDA = 10.0

    total_cost = {
        "A1": (100 - r1) + LAMBDA * c1,
        "A2": (100 - r2) + LAMBDA * c2,
        "A2c": (100 - r2c) + LAMBDA * c2c,
        "A3": (100 - r3) + LAMBDA * c3,
        "A4": (100 - r4) + LAMBDA * c4,
    }
    best_action = min(total_cost, key=total_cost.get)

    rows.append(
        {
            "Condicion": DRIFT_LABEL[drift_type],
            "Sev.": sev_label,
            "A1 Rec%": r1,
            "A1 Costo": f"{c1:.2f}x",
            "A2 Rec%": r2,
            "A2 Costo": f"{c2:.2f}x",
            "A2c Rec%": r2c,
            "A2c Costo": f"{c2c:.2f}x",
            "A3 Rec%": r3,
            "A3 Costo": f"{c3:.2f}x",
            "A4 Rec%": r4,
            "A4 Costo": f"{c4:.2f}x",
            "Accion optima": best_action,
            "_drift_type": drift_type,
            "_sev": sev,
        }
    )

df_results = pd.DataFrame(rows)
print("\nExperimento completado.")

Set de reentrenamiento: 1,122,299 filas

[Covariate shift — Leve (sev=0.2)]
  A1 Defer       : AUC=0.5898  t=0.00s
  A2 Feat update : AUC=0.6109  t=9.92s
  A2 Corrective  : AUC=0.6111  t=0.53s
  A3 Model update: AUC=0.6085  t=86.29s
  A4 Retrain all : AUC=0.6180  t=95.54s

[Covariate shift — Moderada (sev=0.5)]
  A1 Defer       : AUC=0.5556  t=0.00s
  A2 Feat update : AUC=0.5780  t=16.69s
  A2 Corrective  : AUC=0.5777  t=1.45s
  A3 Model update: AUC=0.5983  t=110.89s
  A4 Retrain all : AUC=0.6009  t=102.98s

[Covariate shift — Severa (sev=0.8)]
  A1 Defer       : AUC=0.5425  t=0.00s
  A2 Feat update : AUC=0.5592  t=8.01s
  A2 Corrective  : AUC=0.5590  t=0.60s
  A3 Model update: AUC=0.5960  t=85.12s
  A4 Retrain all : AUC=0.5950  t=93.24s

[Concept drift — Leve (sev=0.2)]
  A1 Defer       : AUC=0.6382  t=0.00s
  A2 Feat update : AUC=0.6428  t=12.32s
  A2 Corrective  : AUC=0.6428  t=0.75s
  A3 Model update: AUC=0.6423  t=183.28s
  A4 Retrain all : AUC=0.6428  t=87.51s

[Concept drift — M

## 8. Tabla Comparativa de Estrategias

**Rec%** = recuperación de AUC respecto al baseline pre-drift.
**Costo** = tiempo relativo medido contra A4 por condición (1.00×).
**A2c** = corrección marginal hacia referencia + Stage 1 congelado + `model_ref` congelado.
**Acción óptima** = menor costo total residual + cómputo.


In [10]:
display_cols = [
    "Condicion",
    "Sev.",
    "A1 Rec%",
    "A1 Costo",
    "A2 Rec%",
    "A2 Costo",
    "A2c Rec%",
    "A2c Costo",
    "A3 Rec%",
    "A3 Costo",
    "A4 Rec%",
    "A4 Costo",
    "Accion optima",
]

COLOR_MAP = {
    "A1": "#006d19",  # verde claro
    "A2": "#013369",  # azul claro
    "A2c": "#006A7D",  # cyan claro
    "A3": "#856600",  # amarillo claro
    "A4": "#6f0009",  # rojo claro
}


def rec_color(val):
    """Colorea celdas Rec% por valor: rojo < 70, amarillo 70-90, verde > 90."""
    if not isinstance(val, (int, float)):
        return ""
    if val >= 90:
        return "background-color: #006d19"
    elif val >= 75:
        return "background-color: #856600"
    else:
        return "background-color: #6f0009"


def optimal_color(val):
    """Colorea la columna de accion optima."""
    return f'background-color: {COLOR_MAP.get(val, "")}; font-weight: bold'


rec_cols = ["A1 Rec%", "A2 Rec%", "A2c Rec%", "A3 Rec%", "A4 Rec%"]

styled = (
    df_results[display_cols]
    .style.applymap(rec_color, subset=rec_cols)
    .applymap(optimal_color, subset=["Accion optima"])
    .set_caption(
        f"AUC baseline = {AUC_BASE:.4f} | "
        f"Rec% = AUC_estrategia / AUC_baseline × 100 | "
        f"Costo relativo a A4"
    )
    .set_table_styles(
        [
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "11px"),
                    ("color", "#666"),
                ],
            }
        ]
    )
)

display(styled)

,Condicion,Sev.,A1 Rec%,A1 Costo,A2 Rec%,A2 Costo,A2c Rec%,A2c Costo,A3 Rec%,A3 Costo,A4 Rec%,A4 Costo,Accion optima
0,Covariate shift,Leve,91.800000,0.00x,95.000000,0.10x,95.100000,0.01x,94.700000,0.90x,96.100000,1.00x,A2c
1,Covariate shift,Moderada,86.400000,0.00x,89.900000,0.16x,89.900000,0.01x,93.100000,1.08x,93.500000,1.00x,A2c
2,Covariate shift,Severa,84.400000,0.00x,87.000000,0.09x,87.000000,0.01x,92.700000,0.91x,92.600000,1.00x,A2c
3,Concept drift,Leve,99.300000,0.00x,100.000000,0.14x,100.000000,0.01x,99.900000,2.09x,100.000000,1.00x,A2c
4,Concept drift,Moderada,89.500000,0.00x,89.500000,0.04x,89.500000,0.00x,92.500000,0.74x,92.500000,1.00x,A1
5,Concept drift,Severa,72.700000,0.00x,68.800000,0.08x,68.900000,0.01x,100.000000,1.57x,100.000000,1.00x,A4
6,Covariate + Concept,Moderada,88.000000,0.00x,88.100000,0.07x,88.100000,0.01x,93.000000,14.42x,92.500000,1.00x,A2c
7,Covariate + Concept,Severa,84.100000,0.00x,84.800000,0.11x,84.900000,0.01x,92.000000,1.60x,92.200000,1.00x,A2c


## 9. Visualización: Rec% por condición y estrategia


In [ ]:
COLORS = {
    "A1": "#6c757d",
    "A2": "#0d6efd",
    "A2c": "#20c997",
    "A3": "#fd7e14",
    "A4": "#198754",
}
LABELS = {
    "A1": "A1 — Defer",
    "A2": "A2 — Stage 1 refit",
    "A2c": "A2c — Corrective",
    "A3": "A3 — Modelo",
    "A4": "A4 — Todo",
}
SERIES = [
    ("A1", "A1 Rec%"),
    ("A2", "A2 Rec%"),
    ("A2c", "A2c Rec%"),
    ("A3", "A3 Rec%"),
    ("A4", "A4 Rec%"),
]

df_plot = df_results.copy()
df_plot["label"] = df_plot["Condicion"] + " — " + df_plot["Sev."]
labels = df_plot["label"].tolist()
n = len(labels)
x = np.arange(n)
w = 0.14

fig, ax = plt.subplots(figsize=(max(12, n * 1.6), 6))

for i, (key, col) in enumerate(SERIES):
    offset = (i - (len(SERIES) - 1) / 2) * w
    ax.bar(
        x + offset,
        df_plot[col],
        w,
        label=LABELS[key],
        color=COLORS[key],
        alpha=0.85,
        edgecolor="white",
    )

ax.axhline(100, color="black", linestyle="--", lw=1, label="Baseline (100%)")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Rec% (AUC recuperado vs baseline)", fontsize=11)
ax.set_title(
    "Recuperación de AUC por estrategia y tipo de drift\n"
    f"(Physionet | Logistic Regression | baseline AUC={AUC_BASE:.3f})",
    fontweight="bold",
    fontsize=12,
)
ax.legend(loc="lower left", fontsize=9)
ax.set_ylim(0, 115)
ax.grid(True, axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## 10. Visualización de drift en VITALS (severidad Moderada)

Efecto del covariate drift (sev=0.5) sobre las distribuciones de los signos vitales.


In [ ]:
df_d_vis, _ = inj.transform(
    df_target,
    drift_severity=0.5,
    numeric_drift_config=NUM_CFG,
    drift_type="covariate",
)

fig = plot_numeric_drift_grid(df_target, df_d_vis, cols=VITALS, ncols=3, bins=60)
plt.suptitle(
    "Covariate drift (sev=0.5) — VITALS antes vs despues",
    fontsize=13,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

## 11. Conclusiones

La tabla resume el patrón esperado del framework **DARL**:

| Tipo de drift   | Etapa afectada          | Estrategia óptima esperada  |
| --------------- | ----------------------- | --------------------------- |
| Covariate shift | Stage 1 — Preprocessing | **A2** (reajustar features) |
| Concept drift   | Stage 2 — Modelo        | **A3** (reentrenar modelo)  |
| Ambos           | Stage 1 + Stage 2       | **A4** (reentrenar todo)    |

El objetivo de DARL es **diagnosticar automáticamente** el tipo de drift
para seleccionar la estrategia óptima sin intervención manual,
minimizando el costo de actualización mientras se maximiza la recuperación de rendimiento.
